# Banking analytics EDA
Synthetic portfolio, seed 81. Run the pipeline first. Open this notebook from the notebooks folder. Tables use INR and calendar year 2025.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
customers = pd.read_csv(ROOT/'data/clean/customers.csv')
loans = pd.read_csv(ROOT/'data/clean/loans.csv')
transactions = pd.read_csv(ROOT/'data/clean/transactions.csv')
customers.describe(include='all')

## Data quality
Occupation Unknown represents a missing source category; unpaid payment dates are intentionally blank. Invalid transactions are quarantined.

In [ ]:
import json
json.loads((ROOT/'outputs/cleaning_audit.json').read_text())

In [ ]:
customers.isna().sum()
transactions.groupby('channel').agg(transactions=('transaction_id','count'),value=('amount','sum'))

## Distribution and relationships
The income/loan relationship and credit score patterns are synthetic. They do not establish causation.

In [ ]:
customers.credit_score.plot.hist(bins=24,title='Credit score distribution'); plt.show()
enriched = loans.merge(customers[['customer_id','annual_income','credit_score']],on='customer_id',validate='many_to_one')
enriched.plot.scatter(x='annual_income',y='original_principal',alpha=.3,title='Annual income vs original principal'); plt.show()

## Snapshot analysis
Use one month-end for balances. Never sum all month-end principal balances into annual outstanding.

In [ ]:
snapshots = pd.read_csv(ROOT/'data/clean/loan_snapshots.csv')
latest = snapshots.query("snapshot_date == '2025-12-31'").merge(enriched,on='loan_id',validate='one_to_one')
latest['score_band'] = pd.cut(latest.credit_score,bins=[299,649,749,900],labels=['<650','650-749','750+'])
latest['dpd90'] = latest.days_past_due.ge(90)
latest.groupby('score_band',observed=True).agg(loans=('loan_id','count'),dpd90_rate=('dpd90','mean'),outstanding=('outstanding','sum'))

In [ ]:
monthly = pd.read_csv(ROOT/'outputs/sql_results/monthly_performance.csv')
monthly.plot(x='month',y=['deposit_balance','loan_outstanding'],marker='o',title='Month-end balances (INR)'); plt.show()

## SQL results and evidence
The SQL uses CTEs, conditional aggregation and window functions; facts are aggregated before joining.

In [ ]:
import sqlite3
con=sqlite3.connect(ROOT/'outputs/banking.db')
pd.read_sql_query("SELECT channel, COUNT(*) AS transaction_count FROM transactions GROUP BY channel",con)
con.close()
json.loads((ROOT/'outputs/validation.json').read_text())

## Business findings
Read the report for quantified findings, recommendations, cash-based income assumptions and no-cure delinquency limitations.